In [1]:
import os, sys, subprocess, warnings, time as timer
warnings.filterwarnings("ignore")

def install(pkg, import_as=None):
    try:
        __import__(import_as or pkg.replace("-", "_").split("[")[0])
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("scikit-survival", "sksurv")
install("lightgbm")
install("xgboost")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 17.9 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import xgboost as xgb
from sksurv.util import Surv
from sksurv.ensemble import GradientBoostingSurvivalAnalysis, RandomSurvivalForest

horizons    = np.array([12.0, 24.0, 48.0, 72.0])
output_path = "/kaggle/working/submission.csv"
run_mode    = "full"

gbsa_seeds_full = (
    123, 456, 789, 777, 666,
    1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31,
    2024, 2077, 3077, 123456, 654321,
    4640, 841, 7755, 8525, 2701,
    8817, 8864, 4085, 8919, 934,
    4746, 1699, 7401, 7826, 4098,
    2921, 1204, 2752, 8384, 1284,
)
gbsa_seeds_fast = tuple(range(42, 52))

rsf_seeds_full = (
    123, 456, 789, 777, 666,
    1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31,
    2024, 2077, 3077, 123456, 654321,
)
rsf_seeds_fast = tuple(range(42, 47))

xgb_seeds_full = (
    123, 456, 789, 777, 666,
    1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31,
    2024, 2077, 3077, 123456, 654321,
)
xgb_seeds_fast = tuple(range(42, 47))

lgb_seeds_full = (
    123, 456, 789, 777, 666,
    1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31,
    2024, 2077, 3077, 123456, 654321,
    2034, 2035, 2036, 1984, 1991,
    3255, 1011, 6241, 2790, 6847,
    8141, 7752, 432, 906, 6217,
    7785, 1603, 7609, 965, 2506,
)
lgb_seeds_fast = tuple(range(42, 52))

gbsa_seeds = gbsa_seeds_full if run_mode == "full" else gbsa_seeds_fast
rsf_seeds  = rsf_seeds_full  if run_mode == "full" else rsf_seeds_fast
xgb_seeds  = xgb_seeds_full  if run_mode == "full" else xgb_seeds_fast
lgb_seeds  = lgb_seeds_full  if run_mode == "full" else lgb_seeds_fast

inner_weights = {"gbsa": 0.65, "rsf": 0.20, "xgb": 0.15}
inner_sum     = sum(inner_weights.values())
power_cal_24  = 1.1


def find_data_dir():
    candidates = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/wids-global-datathon-2026",
        ".",
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "train.csv")):
            return c
    for root, _, files in os.walk("/kaggle/input"):
        if "train.csv" in files:
            return root
    return "."

data_dir = find_data_dir()
print(f"data dir: {data_dir}")

train_df   = pd.read_csv(f"{data_dir}/train.csv")
test_df    = pd.read_csv(f"{data_dir}/test.csv")
sample_sub = pd.read_csv(f"{data_dir}/sample_submission.csv")

print(f"train: {train_df.shape}  test: {test_df.shape}")
print(f"events: {train_df['event'].value_counts().to_dict()}")


def create_features(df):
    r      = df.copy()
    dist   = r["dist_min_ci_0_5h"].clip(lower=1)
    speed  = r["closing_speed_m_per_h"]
    perim  = r["num_perimeters_0_5h"]
    area0  = r["area_first_ha"]
    growth = r["area_growth_rate_ha_per_h"]
    align  = r["alignment_abs"]

    r["dist_km"]              = dist / 1000
    r["log_distance"]         = np.log1p(dist)
    r["inv_distance"]         = 1.0 / (r["dist_km"] + 0.1)
    r["inv_distance_sq"]      = r["inv_distance"] ** 2
    r["sqrt_distance"]        = np.sqrt(dist)
    r["dist_km_sq"]           = r["dist_km"] ** 2
    r["dist_km_cb"]           = r["dist_km"] ** 3
    r["dist_rank"]            = dist.rank(pct=True)

    fire_radius               = np.sqrt(area0 * 10000 / np.pi)
    r["fire_radius_km"]       = fire_radius / 1000
    r["radius_to_dist"]       = fire_radius / dist
    r["area_to_dist_ratio"]   = area0 / (r["dist_km"] + 0.1)
    r["log_area_dist_ratio"]  = np.log1p(area0) - np.log1p(dist)
    r["log1p_area_first_sq"]  = np.log1p(area0) ** 2

    closing_pos               = speed.clip(lower=0)
    r["has_movement"]         = (perim > 1).astype(float)
    r["eta_hours"]            = np.where(closing_pos > 0.01, dist / closing_pos, 9999.0).clip(max=9999)
    r["log_eta"]              = np.log1p(r["eta_hours"].clip(0, 9999))
    r["eta_12h_flag"]         = (r["eta_hours"] <= 12).astype(float)
    r["eta_24h_flag"]         = (r["eta_hours"] <= 24).astype(float)
    r["eta_48h_flag"]         = (r["eta_hours"] <= 48).astype(float)

    radial_g                  = r["radial_growth_rate_m_per_h"].clip(lower=0)
    eff_closing               = closing_pos + radial_g
    r["effective_closing_speed"] = eff_closing
    r["eta_effective"]        = np.where(eff_closing > 0.01, dist / eff_closing, 9999.0).clip(max=9999)
    r["log_eta_effective"]    = np.log1p(r["eta_effective"].clip(0, 9999))
    r["eta_eff_12h_flag"]     = (r["eta_effective"] <= 12).astype(float)
    r["eta_eff_24h_flag"]     = (r["eta_effective"] <= 24).astype(float)
    r["eta_eff_48h_flag"]     = (r["eta_effective"] <= 48).astype(float)
    r["eta_delta"]            = (r["eta_hours"] - r["eta_effective"]).clip(-100, 100)

    r["threat_score"]         = align * speed / np.log1p(dist)
    r["threat_score_sq"]      = r["threat_score"] ** 2
    r["fire_urgency"]         = perim * speed
    r["growth_intensity"]     = growth * perim
    r["momentum"]             = growth * align / (r["dist_km"] + 0.1)
    r["closing_intensity"]    = closing_pos * align
    r["spread_threat"]        = radial_g * align
    r["compound_threat"]      = r["threat_score"] * (dist < 5000).astype(float)

    r["zone_critical"]        = (dist < 5000).astype(float)
    r["zone_warning"]         = ((dist >= 5000)  & (dist < 10000)).astype(float)
    r["zone_moderate"]        = ((dist >= 10000) & (dist < 20000)).astype(float)
    r["zone_safe"]            = (dist >= 20000).astype(float)
    r["critical_threat"]      = r["zone_critical"] * r["threat_score"]
    r["critical_eta"]         = r["zone_critical"] * np.log1p(r["eta_effective"].clip(0, 100))

    r["is_summer"]            = r["event_start_month"].isin([6, 7, 8]).astype(float)
    r["is_autumn"]            = r["event_start_month"].isin([9, 10, 11]).astype(float)
    r["is_afternoon"]         = ((r["event_start_hour"] >= 12) & (r["event_start_hour"] < 20)).astype(float)
    r["is_peak_fire_hour"]    = ((r["event_start_hour"] >= 14) & (r["event_start_hour"] < 18)).astype(float)

    drop = [
        "relative_growth_0_5h", "projected_advance_m",
        "centroid_displacement_m", "centroid_speed_m_per_h",
        "closing_speed_abs_m_per_h", "area_growth_abs_0_5h",
    ]
    r = r.drop(columns=[c for c in drop if c in r.columns])
    r = r.replace([np.inf, -np.inf], np.nan).fillna(0)
    return r


train_proc = create_features(train_df)
test_proc  = create_features(test_df)
print(f"features: {len([c for c in train_proc.columns if c not in ['event_id','event','time_to_hit_hours']])}")


def compute_c_index(time, event, risk):
    n = len(time)
    concordant = comparable = 0
    for i in range(n):
        if event[i] != 1:
            continue
        for j in range(n):
            if i == j or time[i] >= time[j]:
                continue
            comparable += 1
            if   risk[i] > risk[j]:    concordant += 1
            elif risk[i] == risk[j]:   concordant += 0.5
    return concordant / comparable if comparable > 0 else 0.5


def compute_brier(time, event, prob, horizon):
    valid  = ~((event == 0) & (time < horizon))
    if valid.sum() == 0:
        return 0.25
    y_true = ((event == 1) & (time <= horizon)).astype(float)[valid]
    return float(np.mean((np.clip(prob[valid], 0, 1) - y_true) ** 2))


def hybrid_score(time, event, p24, p48, p72):
    risk = 0.3 * p24 + 0.4 * p48 + 0.3 * p72
    c    = compute_c_index(time, event, risk)
    b24  = compute_brier(time, event, p24, 24)
    b48  = compute_brier(time, event, p48, 48)
    b72  = compute_brier(time, event, p72, 72)
    wb   = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * c + 0.7 * (1 - wb), c, wb, b24, b48, b72


def enforce_mono(preds):
    r = np.clip(preds, 0.0, 1.0)
    for i in range(1, r.shape[1]):
        r[:, i] = np.maximum(r[:, i], r[:, i - 1])
    return r


def get_surv_preds(model, x):
    surv_fns = model.predict_survival_function(x)
    preds = np.empty((len(surv_fns), 4), dtype=float)
    for i, fn in enumerate(surv_fns):
        t_min, t_max = fn.domain
        preds[i] = fn(np.clip(horizons, t_min, t_max))
    return 1.0 - preds


def make_binary_target(time_vals, event_vals, horizon):
    unknown = (event_vals == 0) & (time_vals < horizon)
    y       = ((event_vals == 1) & (time_vals <= horizon)).astype(float)
    return y, ~unknown


def compute_ipcw_weights(times, events, horizon):
    unique_t = np.sort(np.unique(times))
    surv     = np.ones(len(unique_t))
    for i, t in enumerate(unique_t):
        at_risk = (times >= t).sum()
        cens_at = ((times == t) & (events == 0)).sum()
        if at_risk > 0:
            surv[i] = 1 - cens_at / at_risk
        if i > 0:
            surv[i] *= surv[i - 1]

    def g(t):
        idx = np.searchsorted(unique_t, t, side="right") - 1
        return max(surv[idx], 0.01) if idx >= 0 else 1.0

    weights = np.ones(len(times))
    for i in range(len(times)):
        if   events[i] == 1 and times[i] <= horizon:
            weights[i] = 1.0 / g(times[i])
        elif times[i] >= horizon:
            weights[i] = 1.0 / g(horizon)
    return weights


def xgb_aft_to_probs(pred_log_time, sigma):
    preds = np.zeros((len(pred_log_time), 4), dtype=float)
    for i, h in enumerate(horizons):
        preds[:, i] = 1.0 / (1.0 + np.exp(-(np.log(h) - pred_log_time) / sigma))
    return preds


x_raw_tr = train_df.drop(columns=["event_id", "event", "time_to_hit_hours"])
x_raw_te = test_df.drop(columns=["event_id"])
x_eng_tr = train_proc.drop(columns=["event_id", "event", "time_to_hit_hours"])
x_eng_te = test_proc.drop(columns=["event_id"])

y_surv     = Surv.from_arrays(event=train_df["event"].astype(bool), time=train_df["time_to_hit_hours"])
event_vals = train_df["event"].values
time_vals  = train_df["time_to_hit_hours"].values
y_lower    = time_vals.astype(float)
y_upper    = np.where(event_vals == 1, time_vals, np.inf).astype(float)


gbsa_configs = [
    {"learning_rate": 0.010, "subsample": 0.70, "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200, "dropout_rate": 0.0},
    {"learning_rate": 0.010, "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 15, "min_samples_split": 3, "n_estimators": 1200, "dropout_rate": 0.0},
    {"learning_rate": 0.010, "subsample": 0.60, "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200, "dropout_rate": 0.0},
    {"learning_rate": 0.005, "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 2000, "dropout_rate": 0.0},
    {"learning_rate": 0.010, "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 20, "min_samples_split": 3, "n_estimators": 1400, "dropout_rate": 0.0},
    {"learning_rate": 0.008, "subsample": 0.75, "max_depth": 2, "min_samples_leaf": 15, "min_samples_split": 4, "n_estimators": 1500, "dropout_rate": 0.0},
    {"learning_rate": 0.015, "subsample": 0.70, "max_depth": 3, "min_samples_leaf": 10, "min_samples_split": 3, "n_estimators": 1000, "dropout_rate": 0.0},
    {"learning_rate": 0.005, "subsample": 0.90, "max_depth": 3, "min_samples_leaf": 18, "min_samples_split": 5, "n_estimators": 2500, "dropout_rate": 0.0},
    {"learning_rate": 0.010, "subsample": 0.80, "max_depth": 4, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200, "dropout_rate": 0.0},
    {"learning_rate": 0.020, "subsample": 0.65, "max_depth": 3, "min_samples_leaf": 10, "min_samples_split": 3, "n_estimators": 800,  "dropout_rate": 0.0},
]

oof_gbsa  = np.zeros((len(x_raw_tr), 4))
test_gbsa = np.zeros((len(x_raw_te), 4))

t0 = timer.time()
print(f"\n[gbsa] {len(gbsa_configs)} configs x {len(gbsa_seeds)} seeds x 5 folds")

for ci, cfg in enumerate(gbsa_configs, 1):
    cfg_oof  = np.zeros_like(oof_gbsa)
    cfg_test = np.zeros_like(test_gbsa)
    for seed in gbsa_seeds:
        s_oof  = np.zeros_like(oof_gbsa)
        s_test = np.zeros_like(test_gbsa)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr_i, va_i in cv.split(x_raw_tr, event_vals):
            m = GradientBoostingSurvivalAnalysis(**{**cfg, "random_state": seed})
            m.fit(x_raw_tr.iloc[tr_i], y_surv[tr_i])
            s_oof[va_i]  = get_surv_preds(m, x_raw_tr.iloc[va_i])
            s_test       += get_surv_preds(m, x_raw_te) / 5
        cfg_oof  += s_oof  / len(gbsa_seeds)
        cfg_test += s_test / len(gbsa_seeds)
    oof_gbsa  += cfg_oof  / len(gbsa_configs)
    test_gbsa += cfg_test / len(gbsa_configs)
    print(f"  config {ci:2d}/{len(gbsa_configs)} done [{(timer.time()-t0)/60:.1f}min]")

print(f"[gbsa] done in {(timer.time()-t0)/60:.1f}min")


rsf_configs = [
    {"n_estimators": 600, "min_samples_leaf": 8,  "max_depth": None, "max_features": "sqrt"},
    {"n_estimators": 600, "min_samples_leaf": 10, "max_depth": None, "max_features": "sqrt"},
    {"n_estimators": 600, "min_samples_leaf": 12, "max_depth": None, "max_features": "sqrt"},
    {"n_estimators": 600, "min_samples_leaf": 8,  "max_depth": 8,   "max_features": "sqrt"},
    {"n_estimators": 600, "min_samples_leaf": 12, "max_depth": 8,   "max_features": "sqrt"},
    {"n_estimators": 800, "min_samples_leaf": 10, "max_depth": None, "max_features": 0.5},
]

oof_rsf  = np.zeros((len(x_raw_tr), 4))
test_rsf = np.zeros((len(x_raw_te), 4))

t0 = timer.time()
print(f"\n[rsf] {len(rsf_configs)} configs x {len(rsf_seeds)} seeds x 5 folds")

for ci, cfg in enumerate(rsf_configs, 1):
    cfg_oof  = np.zeros_like(oof_rsf)
    cfg_test = np.zeros_like(test_rsf)
    for seed in rsf_seeds:
        s_oof  = np.zeros_like(oof_rsf)
        s_test = np.zeros_like(test_rsf)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr_i, va_i in cv.split(x_raw_tr, event_vals):
            m = RandomSurvivalForest(**{**cfg, "random_state": seed, "n_jobs": -1})
            m.fit(x_raw_tr.iloc[tr_i], y_surv[tr_i])
            s_oof[va_i]  = get_surv_preds(m, x_raw_tr.iloc[va_i])
            s_test       += get_surv_preds(m, x_raw_te) / 5
        cfg_oof  += s_oof  / len(rsf_seeds)
        cfg_test += s_test / len(rsf_seeds)
    oof_rsf  += cfg_oof  / len(rsf_configs)
    test_rsf += cfg_test / len(rsf_configs)
    print(f"  config {ci:2d}/{len(rsf_configs)} done [{(timer.time()-t0)/60:.1f}min]")

print(f"[rsf] done in {(timer.time()-t0)/60:.1f}min")


xgb_configs = [
    {"params": {"objective": "survival:aft", "eval_metric": "aft-nloglik", "aft_loss_distribution": "logistic", "aft_loss_distribution_scale": 1.0, "max_depth": 3, "learning_rate": 0.01,  "subsample": 0.70, "colsample_bytree": 0.70, "reg_alpha": 0.5, "reg_lambda": 2.0, "tree_method": "hist", "verbosity": 0}, "n_rounds": 500,  "sigma": 1.0},
    {"params": {"objective": "survival:aft", "eval_metric": "aft-nloglik", "aft_loss_distribution": "logistic", "aft_loss_distribution_scale": 0.8, "max_depth": 3, "learning_rate": 0.01,  "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.1, "reg_lambda": 1.0, "tree_method": "hist", "verbosity": 0}, "n_rounds": 500,  "sigma": 0.8},
    {"params": {"objective": "survival:aft", "eval_metric": "aft-nloglik", "aft_loss_distribution": "logistic", "aft_loss_distribution_scale": 1.2, "max_depth": 2, "learning_rate": 0.02,  "subsample": 0.70, "colsample_bytree": 0.70, "reg_alpha": 1.0, "reg_lambda": 3.0, "tree_method": "hist", "verbosity": 0}, "n_rounds": 400,  "sigma": 1.2},
    {"params": {"objective": "survival:aft", "eval_metric": "aft-nloglik", "aft_loss_distribution": "logistic", "aft_loss_distribution_scale": 0.9, "max_depth": 3, "learning_rate": 0.005, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.3, "reg_lambda": 1.5, "tree_method": "hist", "verbosity": 0}, "n_rounds": 1000, "sigma": 0.9},
    {"params": {"objective": "survival:aft", "eval_metric": "aft-nloglik", "aft_loss_distribution": "logistic", "aft_loss_distribution_scale": 1.1, "max_depth": 2, "learning_rate": 0.01,  "subsample": 0.75, "colsample_bytree": 0.75, "reg_alpha": 0.5, "reg_lambda": 2.5, "tree_method": "hist", "verbosity": 0}, "n_rounds": 600,  "sigma": 1.1},
]

x_xgb_tr = x_eng_tr.values.astype(np.float32)
x_xgb_te = x_eng_te.values.astype(np.float32)

oof_xgb  = np.zeros((len(x_xgb_tr), 4))
test_xgb = np.zeros((len(x_xgb_te), 4))

t0 = timer.time()
print(f"\n[xgb-aft] {len(xgb_configs)} configs x {len(xgb_seeds)} seeds x 5 folds")

for ci, entry in enumerate(xgb_configs, 1):
    p, n_rounds, sigma = entry["params"], entry["n_rounds"], entry["sigma"]
    cfg_oof  = np.zeros_like(oof_xgb)
    cfg_test = np.zeros_like(test_xgb)
    for seed in xgb_seeds:
        ps     = {**p, "seed": seed}
        s_oof  = np.zeros_like(oof_xgb)
        s_test = np.zeros_like(test_xgb)
        dtest  = xgb.DMatrix(x_xgb_te)
        cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr_i, va_i in cv.split(x_xgb_tr, event_vals):
            dtrain = xgb.DMatrix(x_xgb_tr[tr_i])
            dtrain.set_float_info("label_lower_bound", y_lower[tr_i])
            dtrain.set_float_info("label_upper_bound", y_upper[tr_i])
            bst = xgb.train(ps, dtrain, num_boost_round=n_rounds, verbose_eval=False)
            s_oof[va_i] = xgb_aft_to_probs(bst.predict(xgb.DMatrix(x_xgb_tr[va_i])), sigma)
            s_test      += xgb_aft_to_probs(bst.predict(dtest), sigma) / 5
        cfg_oof  += s_oof  / len(xgb_seeds)
        cfg_test += s_test / len(xgb_seeds)
    oof_xgb  += cfg_oof  / len(xgb_configs)
    test_xgb += cfg_test / len(xgb_configs)
    print(f"  config {ci:2d}/{len(xgb_configs)} done [{(timer.time()-t0)/60:.1f}min, sigma={sigma}]")

print(f"[xgb-aft] done in {(timer.time()-t0)/60:.1f}min")


lgb_cfgs_per_h = {
    12: [
        {"max_depth": 2, "learning_rate": 0.03,  "n_estimators": 200, "subsample": 0.70, "colsample_bytree": 0.70, "min_child_samples": 10, "reg_alpha": 1.0, "reg_lambda": 3.0, "num_leaves": 4},
        {"max_depth": 2, "learning_rate": 0.01,  "n_estimators": 400, "subsample": 0.70, "colsample_bytree": 0.70, "min_child_samples": 12, "reg_alpha": 2.0, "reg_lambda": 5.0, "num_leaves": 4},
        {"max_depth": 1, "learning_rate": 0.05,  "n_estimators": 150, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_samples": 8,  "reg_alpha": 0.5, "reg_lambda": 2.0, "num_leaves": 3},
    ],
    24: [
        {"max_depth": 3, "learning_rate": 0.03,  "n_estimators": 300, "subsample": 0.70, "colsample_bytree": 0.70, "min_child_samples": 8,  "reg_alpha": 0.5, "reg_lambda": 2.0, "num_leaves": 7},
        {"max_depth": 2, "learning_rate": 0.02,  "n_estimators": 400, "subsample": 0.75, "colsample_bytree": 0.75, "min_child_samples": 10, "reg_alpha": 1.0, "reg_lambda": 3.0, "num_leaves": 5},
        {"max_depth": 3, "learning_rate": 0.05,  "n_estimators": 200, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_samples": 6,  "reg_alpha": 0.2, "reg_lambda": 1.0, "num_leaves": 7},
    ],
    48: [
        {"max_depth": 2, "learning_rate": 0.05,  "n_estimators": 200, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_samples": 5,  "reg_alpha": 0.1, "reg_lambda": 1.0, "num_leaves": 4},
        {"max_depth": 2, "learning_rate": 0.02,  "n_estimators": 400, "subsample": 0.75, "colsample_bytree": 0.75, "min_child_samples": 8,  "reg_alpha": 0.3, "reg_lambda": 2.0, "num_leaves": 4},
        {"max_depth": 3, "learning_rate": 0.05,  "n_estimators": 200, "subsample": 0.80, "colsample_bytree": 0.70, "min_child_samples": 6,  "reg_alpha": 0.1, "reg_lambda": 0.5, "num_leaves": 6},
    ],
}

data dir: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
train: (221, 37)  test: (95, 35)
events: {0: 152, 1: 69}
features: 72

[gbsa] 10 configs x 40 seeds x 5 folds
  config  1/10 done [4.9min]
  config  2/10 done [10.1min]
  config  3/10 done [14.8min]
  config  4/10 done [23.5min]
  config  5/10 done [29.4min]
  config  6/10 done [35.3min]
  config  7/10 done [39.4min]
  config  8/10 done [50.2min]
  config  9/10 done [55.4min]
  config 10/10 done [58.7min]
[gbsa] done in 58.7min

[rsf] 6 configs x 20 seeds x 5 folds
  config  1/6 done [1.9min]
  config  2/6 done [3.8min]
  config  3/6 done [5.7min]
  config  4/6 done [7.7min]
  config  5/6 done [9.6min]
  config  6/6 done [12.2min]
[rsf] done in 12.2min

[xgb-aft] 5 configs x 20 seeds x 5 folds
  config  1/5 done [0.4min, sigma=1.0]
  config  2/5 done [0.8min, sigma=0.8]
  config  3/5 done [1.0min, sigma=1.2]
  config  4/5 done [1.8min, sigma=0.9]
  config  5/5 done [2.2min, sigma=1.1]
[xgb-aft] done in 2.2min


In [3]:
oof_lgb  = {}
test_lgb = {}

print(f"\n[lgb ipcw] 3 horizons x 3 configs x {len(lgb_seeds)} seeds")

for h in [12, 24, 48]:
    y_bin, mask = make_binary_target(time_vals, event_vals, h)
    valid_idx   = np.where(mask)[0]
    configs     = lgb_cfgs_per_h[h]
    h_oof       = np.zeros(len(x_eng_tr))
    h_test      = np.zeros(len(x_eng_te))

    for cfg in configs:
        a_oof  = np.zeros(len(x_eng_tr))
        a_test = np.zeros(len(x_eng_te))
        for seed in lgb_seeds:
            s_oof  = np.zeros(len(x_eng_tr))
            last_m = None
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
            for tr_v, va_v in cv.split(valid_idx, y_bin[mask]):
                tr_i, va_i = valid_idx[tr_v], valid_idx[va_v]
                w = compute_ipcw_weights(time_vals[tr_i], event_vals[tr_i], h)
                m = lgb.LGBMClassifier(**cfg, objective="binary", random_state=seed, verbose=-1)
                m.fit(x_eng_tr.iloc[tr_i], y_bin[tr_i], sample_weight=w)
                s_oof[va_i] = m.predict_proba(x_eng_tr.iloc[va_i])[:, 1]
                last_m = m
            cens_i = np.where(~mask)[0]
            if len(cens_i) > 0 and last_m is not None:
                s_oof[cens_i] = last_m.predict_proba(x_eng_tr.iloc[cens_i])[:, 1]
            w_full = compute_ipcw_weights(time_vals[valid_idx], event_vals[valid_idx], h)
            m_full = lgb.LGBMClassifier(**cfg, objective="binary", random_state=seed, verbose=-1)
            m_full.fit(x_eng_tr.iloc[valid_idx], y_bin[valid_idx], sample_weight=w_full)
            a_test += m_full.predict_proba(x_eng_te)[:, 1]
            a_oof  += s_oof
        h_oof  += (a_oof  / len(lgb_seeds)) / len(configs)
        h_test += (a_test / len(lgb_seeds)) / len(configs)

    oof_lgb[h]  = h_oof
    test_lgb[h] = h_test
    b = compute_brier(time_vals, event_vals, np.clip(oof_lgb[h], 0, 1), h)
    print(f"  lgb {h:2d}h ({len(configs)} configs)  b@{h:2d}h = {b:.5f}")


print(f"\n[grid search] optimizing blend weights...")

best_score = -1.0
best_ws12 = best_ws24 = best_ws48 = 0.0
best_pcal  = power_cal_24

for ws12 in [0.92, 0.94, 0.95, 0.96, 0.97, 0.98]:
    for ws24 in [0.88, 0.90, 0.92, 0.94, 0.95, 0.96]:
        for ws48 in [0.38, 0.41, 0.44, 0.47, 0.50, 0.53]:
            for pcal in [1.00, 1.05, 1.10, 1.15, 1.20]:
                g_cal = oof_gbsa.copy()
                g_cal[:, 1] = np.clip(g_cal[:, 1] ** pcal, 0, 1)

                def col(ws, h_idx, h_key, g):
                    surv = (inner_weights["gbsa"] / inner_sum * g[:, h_idx]
                          + inner_weights["rsf"]  / inner_sum * oof_rsf[:, h_idx]
                          + inner_weights["xgb"]  / inner_sum * oof_xgb[:, h_idx])
                    return ws * surv + (1 - ws) * oof_lgb[h_key]

                p12 = col(ws12, 0, 12, g_cal)
                p24 = col(ws24, 1, 24, g_cal)
                p48 = col(ws48, 2, 48, g_cal)
                p72 = np.ones(len(p12))
                arr = enforce_mono(np.column_stack([p12, p24, p48, p72]))
                s, *_ = hybrid_score(time_vals, event_vals, arr[:, 1], arr[:, 2], arr[:, 3])
                if s > best_score:
                    best_score = s
                    best_ws12, best_ws24, best_ws48, best_pcal = ws12, ws24, ws48, pcal

print(f"  best oof hybrid: {best_score:.5f}")
print(f"  ws12={best_ws12}  ws24={best_ws24}  ws48={best_ws48}  pcal={best_pcal}")

power_cal_24 = best_pcal
oof_gbsa_cal  = oof_gbsa.copy()
oof_gbsa_cal[:, 1] = np.clip(oof_gbsa_cal[:, 1] ** power_cal_24, 0, 1)
test_gbsa_cal = test_gbsa.copy()
test_gbsa_cal[:, 1] = np.clip(test_gbsa_cal[:, 1] ** power_cal_24, 0, 1)


def make_blend(ws12, ws24, ws48, use_oof):
    gb = oof_gbsa_cal if use_oof else test_gbsa_cal
    rf = oof_rsf       if use_oof else test_rsf
    xb = oof_xgb       if use_oof else test_xgb
    lb = oof_lgb       if use_oof else test_lgb

    def col(ws, h_idx, h_key):
        surv = (inner_weights["gbsa"] / inner_sum * gb[:, h_idx]
              + inner_weights["rsf"]  / inner_sum * rf[:, h_idx]
              + inner_weights["xgb"]  / inner_sum * xb[:, h_idx])
        return ws * surv + (1 - ws) * lb[h_key]

    p12 = col(ws12, 0, 12)
    p24 = col(ws24, 1, 24)
    p48 = col(ws48, 2, 48)
    p72 = np.ones(len(p12))
    return enforce_mono(np.column_stack([p12, p24, p48, p72]))


oof_final  = make_blend(best_ws12, best_ws24, best_ws48, use_oof=True)
test_final = make_blend(best_ws12, best_ws24, best_ws48, use_oof=False)

score, c_idx, wb, b24, b48, b72 = hybrid_score(
    time_vals, event_vals, oof_final[:, 1], oof_final[:, 2], oof_final[:, 3])
b12 = compute_brier(time_vals, event_vals, oof_final[:, 0], 12)

print(f"\n{'='*55}")
print(f"oof hybrid : {score:.5f}")
print(f"  c-index  : {c_idx:.4f}")
print(f"  wbrier   : {wb:.5f}")
print(f"  b12={b12:.5f}  b24={b24:.5f}  b48={b48:.5f}  b72={b72:.5f}")
print(f"{'='*55}")

print(f"\nmodel contributions (standalone oof):")
for name, arr in [("gbsa", oof_gbsa), ("rsf", oof_rsf), ("xgb", oof_xgb)]:
    a = enforce_mono(np.column_stack([arr[:, 0], arr[:, 1], arr[:, 2], np.ones(len(arr))]))
    s, c, wb2, *_ = hybrid_score(time_vals, event_vals, a[:, 1], a[:, 2], a[:, 3])
    print(f"  {name:6s}: hybrid={s:.5f}  c={c:.4f}  wb={wb2:.5f}")
lgb_arr = enforce_mono(np.column_stack([oof_lgb[12], oof_lgb[24], oof_lgb[48], np.ones(len(oof_lgb[12]))]))
s, c, wb2, *_ = hybrid_score(time_vals, event_vals, lgb_arr[:, 1], lgb_arr[:, 2], lgb_arr[:, 3])
print(f"  lgb   : hybrid={s:.5f}  c={c:.4f}  wb={wb2:.5f}")


assert (test_final[:, 1] >= test_final[:, 0]).all()
assert (test_final[:, 2] >= test_final[:, 1]).all()
assert (test_final[:, 3] >= test_final[:, 2]).all()
assert (test_final >= 0).all() and (test_final <= 1).all()

submission = pd.DataFrame({
    "event_id": test_df["event_id"].values,
    "prob_12h":  test_final[:, 0],
    "prob_24h":  test_final[:, 1],
    "prob_48h":  test_final[:, 2],
    "prob_72h":  test_final[:, 3],
})
submission = sample_sub[["event_id"]].merge(submission, on="event_id", how="left")
submission.to_csv(output_path, index=False)

print(f"\nsaved: {output_path}")
print(submission.head(10).to_string(index=False))



[lgb ipcw] 3 horizons x 3 configs x 40 seeds
  lgb 12h (3 configs)  b@12h = 0.05581
  lgb 24h (3 configs)  b@24h = 0.02664
  lgb 48h (3 configs)  b@48h = 0.01038

[grid search] optimizing blend weights...
  best oof hybrid: 0.97157
  ws12=0.92  ws24=0.88  ws48=0.38  pcal=1.2

oof hybrid : 0.97157
  c-index  : 0.9415
  wbrier   : 0.01553
  b12=0.05190  b24=0.03426  b48=0.01313  b72=0.00000

model contributions (standalone oof):
  gbsa  : hybrid=0.97349  c=0.9447  wb=0.01417
  rsf   : hybrid=0.94548  c=0.9321  wb=0.04878
  xgb   : hybrid=0.90372  c=0.9352  wb=0.10980
  lgb   : hybrid=0.97080  c=0.9328  wb=0.01289

saved: /kaggle/working/submission.csv
 event_id  prob_12h  prob_24h  prob_48h  prob_72h
 10662602  0.022863  0.032729  0.032729       1.0
 13353600  0.546692  0.773549  0.915422       1.0
 13942327  0.021934  0.031079  0.031079       1.0
 16112781  0.516552  0.760446  0.912174       1.0
 17132808  0.114503  0.114503  0.114503       1.0
 17445696  0.017715  0.023164  0.023164  